In [ ]:
import requests
import http.client
import pandas as pd
import time
import json
import math
from datetime import datetime, timezone


# ============================================================
# 1. API CONFIGURATION
# ============================================================

#API_KEY = "YOUR_API_KEY"
API_KEY = '49e9776f-878e-43b9-9e7f-8ca23b58de15'

URL = f"https://ca.jooble.org/api/{API_KEY}"

headers = {"Content-Type": "application/json"}


# ============================================================
# 2. RESEARCH PARAMETERS
# ============================================================

positions = [
    "Data Analyst",
    "Data Scientist",
    "Data Engineer"
]

cities = [
    "Toronto, ON",
    "Vancouver, BC",
    "Montreal, QC"
]

RADIUS = "26"

# Number of jobs requested per page
RESULTS_PER_PAGE = 100

# Maximum number of pages allowed per position-city combination
# CHANGE THIS TO 20 TO GET 2000 JOB POSTING PER EACH POSITION X CITY
MAX_PAGES = 25

# Delay between API requests
REQUEST_DELAY = 2.5


# ============================================================
# 3. STORAGE
# ============================================================

all_jobs = []


# ============================================================
# 4. LOOP THROUGH POSITIONS AND CITIES
# ============================================================

for position in positions:

    for city in cities:

        print("\n" + "=" * 70)
        print(f"Starting search: {position} | {city}")
        print("=" * 70)

        # ----------------------------------------------------
        # FIRST REQUEST
        # Used to get totalCount and page 1 results
        # ----------------------------------------------------

        first_payload = {
            "keywords": position,
            "location": city,
            "radius": RADIUS,
            "page": 12,
            "ResultOnPage": RESULTS_PER_PAGE
        }

        try:

            response = requests.post(
                URL,
                headers=headers,
                json=first_payload,
                timeout=30
            )

            response.raise_for_status()

            data = response.json()

            total_jobs = data.get("totalCount", 0)
            jobs = data.get("jobs", [])

            print(f"Total jobs reported by Jooble: {total_jobs}")

            # Calculate the number of pages actually needed
            pages_needed = math.ceil(total_jobs / RESULTS_PER_PAGE)

            # Never exceed MAX_PAGES
            pages_to_collect = min(pages_needed, MAX_PAGES)

            print(f"Pages needed: {pages_needed}")
            print(f"Pages that will be collected: {pages_to_collect}")

            # ------------------------------------------------
            # SAVE PAGE 1
            # ------------------------------------------------

            request_datetime = datetime.now(timezone.utc).isoformat()

            for job in jobs:

                job["search_position"] = position
                job["search_city"] = city
                job["request_datetime"] = request_datetime
                job["source_site"] = "Jooble"

                all_jobs.append(job)

            print(
                f"{position} | {city} | "
                f"Page 1 | Jobs retrieved: {len(jobs)}"
            )

            # If only one page is needed, skip directly
            # to the next city/position combination
            if pages_to_collect <= 1:
                continue

            time.sleep(REQUEST_DELAY)


            # ------------------------------------------------
            # REQUEST REMAINING PAGES
            # ------------------------------------------------

            for page in range(13, pages_to_collect + 1):

                payload = {
                    "keywords": position,
                    "location": city,
                    "radius": RADIUS,
                    "page": page,
                    "ResultOnPage": RESULTS_PER_PAGE
                }

                try:

                    response = requests.post(
                        URL,
                        headers=headers,
                        json=payload,
                        timeout=30
                    )

                    response.raise_for_status()

                    page_data = response.json()

                    jobs = page_data.get("jobs", [])

                    print(
                        f"{position} | {city} | "
                        f"Page {page} | Jobs retrieved: {len(jobs)}"
                    )

                    # Extra safety:
                    # stop if the API unexpectedly returns no jobs
                    if len(jobs) == 0:
                        print("No more jobs returned. Stopping early.")
                        break

                    request_datetime = datetime.now(
                        timezone.utc
                    ).isoformat()

                    for job in jobs:

                        job["search_position"] = position
                        job["search_city"] = city
                        job["request_datetime"] = request_datetime
                        job["source_site"] = "Jooble"

                        all_jobs.append(job)

                    time.sleep(REQUEST_DELAY)

                except requests.exceptions.RequestException as e:

                    print(
                        f"API error for {position} | "
                        f"{city} | Page {page}: {e}"
                    )

                    continue


        except requests.exceptions.RequestException as e:

            print(
                f"Initial API request failed for "
                f"{position} | {city}: {e}"
            )

            continue


# ============================================================
# 5. CREATE DATAFRAME
# ============================================================

df_jooble = pd.DataFrame(all_jobs)


# ============================================================
# 6. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("COLLECTION COMPLETE")
print("=" * 70)

print(f"Total jobs collected: {len(df_jooble)}")
print(f"Total columns: {len(df_jooble.columns)}")

if not df_jooble.empty:

    print("\nJobs collected by position and city:")

    print(
        df_jooble.groupby(
            ["search_position", "search_city"]
        ).size()
    )


# ============================================================
# 7. SAVE RAW DATA
# ============================================================

output_file = "jooble_raw_jobs.csv"

df_jooble.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nCSV saved successfully: {output_file}")


Starting search: Data Analyst | Toronto, ON
Total jobs reported by Jooble: 13898
Pages needed: 139
Pages that will be collected: 25
Data Analyst | Toronto, ON | Page 1 | Jobs retrieved: 16
Data Analyst | Toronto, ON | Page 13 | Jobs retrieved: 0
No more jobs returned. Stopping early.

Starting search: Data Analyst | Vancouver, BC
Total jobs reported by Jooble: 4892
Pages needed: 49
Pages that will be collected: 25
Data Analyst | Vancouver, BC | Page 1 | Jobs retrieved: 0
Data Analyst | Vancouver, BC | Page 13 | Jobs retrieved: 0
No more jobs returned. Stopping early.

Starting search: Data Analyst | Montreal, QC
Total jobs reported by Jooble: 10888
Pages needed: 109
Pages that will be collected: 25
Data Analyst | Montreal, QC | Page 1 | Jobs retrieved: 0
Data Analyst | Montreal, QC | Page 13 | Jobs retrieved: 0
No more jobs returned. Stopping early.

Starting search: Data Scientist | Toronto, ON
Total jobs reported by Jooble: 12917
Pages needed: 130
Pages that will be collected: 25
Da